### INPUT GUARDRAILS

#### Block bad inputs

#### Prevent prompt injection

#### Ensure domain relevance

In [23]:
#!pip install --upgrade crewai crewai_tools langchain_community
#!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
#!pip install --upgrade crewai chromadb
#!pip install langchain-core langchain-openai
#!pip install faiss-cpu
#!pip install pysqlite3-binary
#!pip install -U crewai crewai-tools

###### NEEDED THIS AS CURRENT SESSION APICREDITS WERE EXHAUSTED

In [2]:
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
OPENAI_BASE_URL = "https://openai.vocareum.com/v1"

###### HELPER FUNCTIONS

In [26]:
import json
import re

#STRING TO JSON CONVERTER FUNCTION, IT ALSO CHECKS FOR VALID JSON
def safe_json_parse(raw_output):
    """
    Extracts the first valid JSON object from CrewAI output safely.
    """

    if isinstance(raw_output, dict):
        return raw_output

    # Find JSON block using regex
    match = re.search(r'\{.*\}', raw_output, re.DOTALL)

    if not match:
        raise ValueError("No JSON object found in CrewAI output.")

    json_str = match.group()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format: {e}")

# FUNCTION FOR INCIDENT INPUT VALIDATION
def validate_incident_input(text: str) -> str:
    if not text or len(text.strip()) < 15:
        raise ValueError("Incident description too short.")

    forbidden = [
        "ignore previous instructions",
        "override policy",
        "do not escalate",
        "act as"
    ]

    for phrase in forbidden:
        if phrase in text.lower():
            raise ValueError("Unsafe instruction detected.")

    domain_keywords = [
        "shipment", "delay", "inventory", "vendor",
    "system", "warehouse", "delivery", "production",
    "stock", "storage"
    ]

    if not any(k in text.lower() for k in domain_keywords):
        raise ValueError("Non–supply-chain incident.")

    return text

### RAG KNOWLEDGE BASE (REAL POLICY DOCUMENTS)

In [4]:
from langchain.docstore.document import Document

policy_docs = [
    Document(
        page_content="""
        SLA POLICY

        Tier 1 clients allow a maximum shipment delay of 6 hours.
        Tier 2 clients allow a maximum shipment delay of 24 hours.
        Any incident impacting production continuity must be escalated.
        Repeated SLA breaches within 7 days require escalation.
        """,
        metadata={"policy_type": "SLA"}
    ),

    Document(
        page_content="""
        VENDOR CONTRACT POLICY

        Failure to deliver critical components is a P1 incident.
        More than two vendor delivery failures within 7 days require escalation.
        Vendor issues affecting production must always be escalated.
        """,
        metadata={"policy_type": "Vendor"}
    ),

    Document(
        page_content="""
        INCIDENT HANDLING SOP

        Incidents with classification confidence below 0.6 must be reviewed manually.
        System outages below 30 minutes may be auto-resolved if fully restored.
        Inventory mismatches below five units may be auto-resolved.
        """,
        metadata={"policy_type": "SOP"}
    )
]

### VECTOR STORE (RAG CORE)

In [5]:
from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS 
from langchain_core.documents import Document 

embeddings = OpenAIEmbeddings(
    api_key= OPENAI_API_KEY,
    base_url= OPENAI_BASE_URL
)

vectorstore = FAISS.from_documents(
    policy_docs,
    embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### RAG GUARDRAILS (ANTI-HALLUCINATION)

In [6]:
def validate_rag_retrieval(docs):
    if not docs or len(docs) == 0:
        raise ValueError("No policy context retrieved. Escalation required.")

### LLM SETUP

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",   # or whichever model you're using
    temperature=0.2,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL
)

### AGENTS (ALL RAG-AWARE)

In [10]:
from crewai import Agent
from crewai_tools import RagTool

rag_tool = RagTool(
    name="Knowledge Base",
    description="Useful for retrieving relevant policy documents. Input should be a single 'query' string to search for policy details.",
    retriever=retriever
)

2026-02-27 07:34:26,535 [embedchain] [INFO] Swapped std-lib sqlite3 with pysqlite3 for ChromaDb compatibility. Your original version was 3.31.1.
/voc/work/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:918: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warn(
2026-02-27 07:34:28,370 - 139621583755072 - posthog.py-posthog:59 - ERROR: Failed to send telemetry event ClientStartEvent: module 'chromadb' has no attribute 'get_settings'
2026-02-27 07:34:28,425 - 139621583755072 - posthog.py-posthog:59 - ERROR: Failed to send telemetry event ClientCreateCollectionEvent: module 'chromadb' has no attribute 'get_settings'
2026-02-27 07:34:28,465 - 139621583755072 - posthog.py-posthog:59 - ERROR: Failed to send telemetry event ClientCreateCollectionEvent: module 'chromadb' has no attribute 'get_settings'


## Agent 1 - Incident Classification Agent

In [11]:
incident_classifier = Agent(
    role="Incident Classification Agent",
    goal="Extract structured incident signals with confidence estimation",
    backstory="You classify supply chain incidents without applying policy rules.",
    llm=llm,
    #verbose=True,
    verbose=False,
    max_iterations=2
)

## Agent 2 - Policy Intelligence Agent

In [12]:
policy_agent = Agent(
    role="Policy Intelligence Agent",
    goal="Apply retrieved policy documents to assess risk and impact",
    backstory=(
        "You MUST base decisions strictly on retrieved documents. "
        "If insufficient information exists, state INSUFFICIENT_POLICY_CONTEXT."
    ),
    tools=[rag_tool],
    llm=llm,
    #verbose=True,
    verbose=False,
    max_iterations=3
)

## Agent 3 - Action Recommendation Agent

In [13]:
action_agent = Agent(
    role="Action Recommendation Agent",
    goal="Recommend a response grounded in policy findings",
    backstory="You recommend actions but do not decide escalation.",
    llm=llm,
    #verbose=True,
    verbose=False,
    max_iterations=3
)

## Agent 4 - Escalation Decision Agent

In [14]:
escalation_agent = Agent(
    role="Escalation & Priority Decision Agent",
    goal="Make the final escalation decision using policy and guardrails",
    backstory="You prioritize safety and escalate when uncertainty exists.",
    llm=llm,
    #verbose=True,
    verbose=False,
    max_iterations=3
)

### TASKS

In [15]:
from crewai import Task

#### Task 1 - Classification

In [16]:
classification_task = Task(
    description="""
Analyze the incident and output JSON:
{{
  "incident_type": "...",
  "severity": "...",
  "confidence_score": 0.0-1.0
}}
Incident: {incident_text}

You MUST return the final answer immediately.
Do NOT explain your reasoning.
Do NOT ask follow-up questions.
Return ONLY the JSON object.
""",
    expected_output="""
A valid JSON object containing:
- incident_type
- severity
- confidence_score
""",
    agent=incident_classifier
)

#### Task 2 - RAG Policy Reasoning

In [17]:
policy_task = Task(
    description="""
Retrieve relevant policy documents and determine:
{{
  "sla_breach_risk": true/false,
  "business_impact": "low|medium|high",
  "policy_justification": "quoted reasoning"
}}

You MUST return the final answer immediately.
Do NOT explain your reasoning.
Do NOT ask follow-up questions.
Return ONLY the JSON object.

If policies are insufficient, respond with:
INSUFFICIENT_POLICY_CONTEXT
""",
    expected_output="""
A JSON object containing:
- sla_breach_risk
- business_impact
- policy_justification
""",
    agent=policy_agent
)

#### Task 3 - Action Recommendation

In [18]:
action_task = Task(
    description="""
Based on classification and policy reasoning, output STRICT JSON:

{{
  "recommended_action": "auto_resolve | monitor | escalate",
  "rationale": "short explanation"
}}

You MUST return the final answer immediately.
Do NOT explain your reasoning.
Do NOT ask follow-up questions.
Return ONLY the JSON object.
""",
    expected_output="""
A JSON object containing:
- recommended_action
- rationale
""",
    agent=action_agent
)

#### Task 4 - Final Escalation Decision

In [19]:
escalation_task = Task(
    description="""
Make final decision and output STRICT JSON:

{{
  "final_decision": "auto_resolve | escalate",
  "priority": "P1 | P2 | P3 | none",
  "justification": "clear reasoning"
}}

You MUST return the final answer immediately.
Do NOT explain your reasoning.
Do NOT ask follow-up questions.
Return ONLY the JSON object.

Escalate if:
- confidence_score < 0.6
- sla_breach_risk == true
- business_impact == high
- policy context insufficient
""",
    expected_output="""
A JSON object containing:
- final_decision
- priority
- justification
""",
    agent=escalation_agent
)

## CREW ASSEMBLY

In [20]:
from crewai import Crew, Process

crew = Crew(
    agents=[
        incident_classifier,
        policy_agent,
        action_agent,
        escalation_agent
    ],
    tasks=[
        classification_task,
        policy_task,
        action_task,
        escalation_task
    ],
    process=Process.sequential,
    verbose=False
)

## FINAL PIPELINE EXECUTION STEP

In [21]:
def run_incident_pipeline(raw_text):
    text = validate_incident_input(raw_text)
    print(text)
    # Run the crew
    result = crew.kickoff(inputs={"incident_text": text})
    print(result)

    # result.json_dict contains the output of the LAST task
    # To get the classification from Task 1:
    #classification = result.tasks_output[0].json_dict
    #final_decision = result.tasks_output[3].json_dict
    final_result = safe_json_parse(result)

    return final_result

'def run_incident_pipeline(raw_text):\n    # Input guardrail\n    text = validate_incident_input(raw_text)\n\n    # Run CrewAI\n    raw_result = crew.kickoff(\n        inputs={"incident_text": text}\n    )\n\n    # Parse final output safely\n    #final_result = safe_json_parse(raw_result)\n\n    return final_result'

### EXAMPLE RUN

In [22]:
test_incidents = [
    "Shipment to Delhi DC delayed by 12 hours due to heavy rain.",
    "Inventory mismatch of 3 units found during warehouse scan.",
    "Vendor X failed to deliver critical components; production at risk.",
    "Warehouse system down for 20 minutes; restored now.",
    "Vendor Y has delayed shipments three times this week.",
    "Shipment delay of 8 hours for Tier 1 client.",
    "Something seems wrong with stock numbers in storage area."
]

In [23]:
def Report_Incident(incident: str, scenario_number: int = None):
    if scenario_number is not None:
        print("\n" + "="*70)
        print(f"SCENARIO {scenario_number}")
    else:
        print("\n" + "="*70)
        print("SCENARIO")

    print("Incident:", incident)
    print("-"*70)

    try:
        result = run_incident_pipeline(incident)

        print("\nFINAL DECISION OUTPUT:")
        print("Final Decision :", result.get("final_decision", "N/A"))
        print("Priority       :", result.get("priority", "N/A"))
        print("Justification  :", result.get("justification", "N/A"))

    except Exception as e:
        print("\nGUARDRAIL TRIGGERED:")
        print(str(e))

## TEST INCIDENT 1

In [24]:
Report_Incident(test_incidents[2])


SCENARIO
Incident: Vendor X failed to deliver critical components; production at risk.
----------------------------------------------------------------------
Vendor X failed to deliver critical components; production at risk.
 

The appropriate severity level for the incident involving Vendor X's failure to deliver critical components is categorized as "High Severity." This classification is due to the direct impact on our production process, which is at risk of significant operational disruptions. The inability to receive these essential components can lead to delays in production timelines, potential financial losses, and a negative effect on our overall supply chain. Given the critical nature of these components, it is imperative to escalate this issue to senior management and the procurement team immediately to explore alternative solutions, such as sourcing from backup vendors or negotiating expedited delivery options. Additionally, we should initiate a risk assessment to evaluat

## TEST INCIDENT 2

In [45]:
Report_Incident(test_incidents[1])


SCENARIO 1
Incident: Inventory mismatch of 3 units found during warehouse scan.
----------------------------------------------------------------------
Inventory mismatch of 3 units found during warehouse scan.
 

The severity level for an inventory mismatch of 3 units found during a warehouse scan can be classified as a "Moderate Severity" issue. This classification is based on the following considerations: 

1. **Impact on Operations**: A mismatch of 3 units may not significantly disrupt overall operations, especially if the total inventory is large. However, it can indicate potential issues in inventory management practices that need to be addressed.

2. **Potential for Escalation**: While 3 units may seem minor, it is important to investigate the root cause of the discrepancy. This could be a sign of larger systemic issues such as theft, data entry errors, or mislabeling, which could escalate if not addressed.

3. **Financial Implications**: Depending on the value of the items invo

## TEST INCIDENT 3

In [25]:
Report_Incident(test_incidents[3])


SCENARIO
Incident: Warehouse system down for 20 minutes; restored now.
----------------------------------------------------------------------
Warehouse system down for 20 minutes; restored now.
 

The severity level for a warehouse system downtime of 20 minutes that has been restored can be classified as a Severity Level 2 incident. This classification is based on the impact of the downtime on operations and the fact that the system has been restored without any long-term effects. While 20 minutes of downtime can disrupt operations and may lead to minor delays in order processing or inventory management, it does not typically result in significant financial loss or safety concerns. Therefore, it is important to document the incident, analyze the root cause to prevent future occurrences, and ensure that all stakeholders are informed. Additionally, implementing measures to minimize downtime in the future would be beneficial for operational efficiency.

 

I encountered an error while tr